# 1. Generate a small labelled benchmark dataset with Amazon Nova 2 Lite

> **Note:** This is sample code for demonstration purposes only and is not intended for production use without
> additional security testing and review.

CLM answers two kinds of request, so the dataset has two parts:

| Part | CLM call | What Nova 2 Lite writes | Gold label |
|---|---|---|---|
| `tickets` | `systemone` (typed questions about a state) | a short customer-support message | department, urgent yes/no, frustration 0-2 |
| `rank` | `rank` (pick the best of N free-form candidates) | a context + question, 1 correct and 7 plausible wrong candidates | index of the correct candidate |

The labels let notebook 2 check that the deployed endpoint gives *correct* answers, not just fast ones.
The output file stays under 100 KB: enough requests for stable latency and throughput numbers.

The committed `data/clm_bench.jsonl` was produced by this notebook. You only need to rerun it to regenerate the data.

The tickets are synthetic and contain no real customer data. If you build a dataset from real customer messages
instead, treat it as personal data: remove or mask identifiers before sending it to any model, and follow the rules
that apply to you (for example GDPR, HIPAA or PCI DSS).

The prompts here are fixed and the output is checked against a schema, so the calls run without a guardrail. If you
let users supply the prompts, or keep generated text that people will read, attach an
[Amazon Bedrock Guardrail](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails.html) to the
`converse` call (`guardrailConfig`) to filter harmful content and personal data.

In [1]:
import json, re, collections
from pathlib import Path
import boto3, numpy as np
from botocore.config import Config

REGION = "us-east-1"                      # the same Region as notebook 2
MODEL_ID = "us.amazon.nova-2-lite-v1:0"   # Nova 2 Lite, US cross-Region inference profile
OUT = Path("data/clm_bench.jsonl")
DEPARTMENTS = ["billing", "technical", "account", "shipping", "sales"]

# A retry with a doubled token limit can take minutes to generate, past botocore's 60 s default read timeout.
brt = boto3.client("bedrock-runtime", region_name=REGION, config=Config(read_timeout=900))

def nova_json(prompt: str, temperature: float = 0.8) -> list:
    """Ask Nova 2 Lite for a JSON array and parse it (tolerates a ```json fence); retry on bad JSON.

    A reply cut off at the token limit is retried with twice the limit rather than the same one."""
    max_tokens, cut_off = 6000, 0
    for attempt in range(3):
        r = brt.converse(modelId=MODEL_ID,
                         messages=[{"role": "user", "content": [{"text": prompt}]}],
                         inferenceConfig={"maxTokens": max_tokens, "temperature": temperature})
        text = r["output"]["message"]["content"][0]["text"]
        if r.get("stopReason") == "max_tokens":
            print(f"  attempt {attempt + 1}: the reply hit {max_tokens} tokens, retrying with {2 * max_tokens}")
            max_tokens *= 2
            cut_off += 1
            continue
        try:
            return json.loads(re.search(r"\[.*\]", text, re.S).group(0), strict=False)
        except (AttributeError, ValueError) as e:
            print(f"  attempt {attempt + 1}: unparseable JSON ({e}), retrying")
    if cut_off == 3:
        raise RuntimeError(f"every Nova 2 Lite reply was cut off at the token limit (up to {max_tokens // 2}): "
                           "ask for fewer items per call")
    raise RuntimeError("Nova 2 Lite did not return valid JSON in 3 attempts")

## Support tickets (for `systemone`)

In [2]:
TICKET_PROMPT = """Write {n} realistic, varied customer-support messages for a {industry} company.
Each message is 1-3 sentences, written by the customer, no greeting or signature.
Return ONLY a JSON array of objects with keys:
  "text": the message,
  "department": one of {departments} (the team that should handle it),
  "urgent": true or false (true only if the customer is blocked, losing money, or facing a deadline),
  "frustration": 0 (calm), 1 (frustrated) or 2 (very angry).
Spread the departments evenly and vary urgency and tone."""

INDUSTRIES = ["SaaS analytics", "e-commerce fashion", "telecom", "online banking", "airline", "food delivery"]
tickets = []
for industry in INDUSTRIES:
    rows = nova_json(TICKET_PROMPT.format(n=20, industry=industry, departments=DEPARTMENTS))
    # Keep only rows whose labels have exactly the expected types; a string "false" must not become True.
    tickets += [{"text": r["text"].strip(), "department": r["department"], "urgent": r["urgent"],
                 "frustration": r["frustration"]}
                for r in rows if isinstance(r, dict) and isinstance(r.get("text"), str) and r["text"].strip()
                and r.get("department") in DEPARTMENTS and isinstance(r.get("urgent"), bool)
                and type(r.get("frustration")) is int and r["frustration"] in (0, 1, 2)]
print(len(tickets), "tickets;", collections.Counter(t["department"] for t in tickets))

119 tickets; Counter({'account': 31, 'technical': 30, 'billing': 25, 'sales': 22, 'shipping': 11})


## Ranking items (for `rank`)

In [3]:
RANK_PROMPT = """Write {n} multiple-choice items about: {topic}.
Each item has a short "context" (1-2 sentences describing the situation), a "question",
one "correct" answer and exactly 7 "wrong" answers that are plausible but clearly worse.
Every answer is at most 15 words. Return ONLY a JSON array of objects with keys
"context", "question", "correct", "wrong" (a list of 7 strings)."""

TOPICS = ["which tool an AI agent should call next (answers are tool calls like search_flights(...))",
          "the next UI action for a computer-use agent on a web page",
          "fixing a failing unit test or a Python error message",
          "everyday science and geography facts",
          "the best reply a support agent should send to a customer",
          "choosing the right AWS service for a requirement"]
rank = []
for topic in TOPICS:
    rows = nova_json(RANK_PROMPT.format(n=17, topic=topic))
    for r in (r for r in rows if isinstance(r, dict)):
        wrong = r.get("wrong") if isinstance(r.get("wrong"), list) else []
        wrong = [w.strip() for w in wrong if isinstance(w, str) and w.strip()]
        if not all(isinstance(r.get(k), str) and r[k].strip() for k in ("context", "question", "correct")):
            continue
        correct = r["correct"].strip()
        if len(wrong) == 7 and len(set(wrong + [correct])) == 8:     # 8 distinct options, so the gold label is unambiguous
            rank.append({"context": r["context"].strip(), "question": r["question"].strip(),
                         "correct": correct, "wrong": wrong})
print(len(rank), "rank items")

  attempt 1: unparseable JSON (Expecting ':' delimiter: line 73 column 60 (char 1860)), retrying


99 rank items


## Shuffle, dedupe and save
The correct answer's position is shuffled with a fixed seed, so the gold index is not always 0. (A seeded NumPy
generator is enough here: the shuffle only has to be reproducible, not unpredictable.)

In [4]:
rng = np.random.default_rng(0)
seen, rows = set(), []
for t in tickets:
    if t["text"] not in seen:
        seen.add(t["text"])
        rows.append({"kind": "ticket", **t})
for r in rank:
    if r["question"] + r["context"] in seen:
        continue
    seen.add(r["question"] + r["context"])
    answers = [r["correct"], *r["wrong"]]
    answers = [answers[i] for i in rng.permutation(len(answers))]
    rows.append({"kind": "rank", "context": r["context"], "question": r["question"],
                 "answers": answers, "gold": answers.index(r["correct"])})

OUT.parent.mkdir(exist_ok=True)
OUT.write_text("".join(json.dumps(x, ensure_ascii=False, separators=(",", ":")) + "\n" for x in rows), encoding="utf-8")
print(f"{OUT}: {sum(r['kind'] == 'ticket' for r in rows)} tickets, {sum(r['kind'] == 'rank' for r in rows)} rank items, "
      f"{OUT.stat().st_size / 1024:.1f} KB")
print(json.dumps(rows[0], indent=1)[:400])

data/clm_bench.jsonl: 119 tickets, 99 rank items, 47.6 KB
{
 "kind": "ticket",
 "text": "I'm trying to set up a new data source but keep getting an authentication error. Can someone help me troubleshoot this?",
 "department": "technical",
 "urgent": true,
 "frustration": 1
}
